# 06 - Đánh giá cuối cùng trên official test

Notebook này đọc winner đã được khóa bởi notebook 05 và đánh giá đúng
checkpoint đó trên official test. Test không được dùng để điều chỉnh
hyperparameter, chọn lại checkpoint hoặc thay đổi winner

Quy trình gồm nạp `model_selection.json`, nạp checkpoint từ đĩa, đánh
giá toàn bộ `test.jsonl`, tạo confusion matrix, phân tích lỗi và ghi
kết quả vào `outputs/evaluation/`. Winner không được ghi cứng trong
notebook; `load_locked_winner()` từ chối artifact chưa khóa hoặc đã sử
dụng test để chọn model


In [1]:
import os, sys, json, time
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src").exists(), f"Cannot locate the repo root from {Path.cwd()}"
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from src.data.constants import LABELS, PUNCTUATION_LABELS, EXPERIMENT_IDS, OUTPUTS_DIR
from src.utils.io import read_json, write_json, write_csv
from src.utils.logging_utils import configure_stdout_utf8

configure_stdout_utf8()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

EVALUATION_DIR = OUTPUTS_DIR / "evaluation"
FIGURES_DIR = OUTPUTS_DIR / "figures"
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: .


In [ ]:
import torch
from src.evaluation.selection import load_locked_winner, resolve_winner_checkpoint

selection = load_locked_winner()
WINNER = selection["winner"]
CKPT_DIR = resolve_winner_checkpoint(selection)

print("model_selection.json")
for k in ("selection_split", "selection_metric", "tie_breaker", "winner",
          "test_was_used_for_selection", "winner_locked", "locked_at_utc"):
    print(f"  {k:<32}: {selection[k]}")
print()
print(f"Winner            : {WINNER}  ({selection['winner_model']}, "
      f"weight_mode={selection['winner_weight_mode']})")
print(f"Chosen on          : validation Punctuation Macro-F1 = "
      f"{selection['winner_validation_punctuation_macro_f1']:.6f}")
print(f"Checkpoint         : {CKPT_DIR.relative_to(PROJECT_ROOT)}")

model_selection.json
  selection_split                 : validation
  selection_metric                : punctuation_macro_f1
  tie_breaker                     : unweighted_validation_loss
  winner                          : E2
  test_was_used_for_selection     : False
  winner_locked                   : True
  locked_at_utc                   : 2026-08-09T22:59:40+00:00

Winner            : E2  (vinai/phobert-base-v2, weight_mode=none)
Chosen on          : validation Punctuation Macro-F1 = 0.778718
Checkpoint         : outputs\checkpoints\E2


## 1. Nạp checkpoint và dữ liệu test

Checkpoint được lấy từ đường dẫn trong `model_selection.json`. Model và
tokenizer hoặc vocabulary được nạp từ cùng thư mục checkpoint để bảo
đảm cách mã hóa đầu vào khớp với lúc huấn luyện

Sau khi xác nhận winner đã khóa, notebook mới đọc `test.jsonl` và tạo
DataLoader phù hợp với kiến trúc BiLSTM hoặc PhoBERT


In [3]:
from src.data.constants import PROCESSED_FILES
from src.data.dataset import load_examples
from src.evaluation.loaders import build_eval_dataloader
from src.models.factory import load_model_from_checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model, meta = load_model_from_checkpoint(CKPT_DIR, device=device)
print(f"Loaded {meta['model_type']} — {meta['model_name']}")
print(f"  revision   : {meta.get('model_revision')}")
print(f"  best epoch : {meta['best_epoch']}   (validation score {meta['best_score']:.6f})")
print(f"  seed       : {meta['seed']}")
print(f"  label2id   : {meta['label2id']}")


test_examples = load_examples(PROCESSED_FILES["test"])
print(f"\nTest split: {len(test_examples):,} examples, "
      f"{sum(len(e.tokens) for e in test_examples):,} words")

test_loader, model_type, encoder = build_eval_dataloader(
    CKPT_DIR, test_examples, batch_size=32
)
print(f"Batches: {len(test_loader):,}")

Device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded phobert — vinai/phobert-base-v2
  revision   : fb76b7e1f77fa19bc4870e2ad956876f7c81c53f
  best epoch : 5   (validation score 0.778718)
  seed       : 42
  label2id   : {'O': 0, 'COMMA': 1, 'PERIOD': 2, 'QUESTION': 3}


2026-08-10 06:00:34 | INFO    | src.data.dataset | Loaded 22832 examples from test.jsonl



Test split: 22,832 examples, 2,968,815 words
2026-08-10 06:00:34 | INFO    | src.evaluation.loaders | Loaded PhoBERT tokenizer from E2 (max_length=192)


2026-08-10 06:00:38 | INFO    | src.data.dataset | Encoded 20000/22832 examples


2026-08-10 06:00:39 | INFO    | src.data.dataset | PhoBERT dataset ready: {'num_examples': 22832, 'num_windows': 22857, 'num_words': 2968815, 'num_subwords': 3129144, 'windows_per_example': 1.0011, 'subwords_per_word': 1.054, 'examples_split_into_multiple_windows': 25, 'max_window_length': 192}


Batches: 715


## 2. Đánh giá trên official test

Metric được tính tại đúng một vị trí cho mỗi từ; các vị trí mang
`IGNORE_INDEX` bị bỏ qua. Với PhoBERT, chuỗi dài được chia thành nhiều
cửa sổ tại ranh giới từ rồi gộp lại mà không cắt mất dữ liệu

Loss dùng trong lần đánh giá là cross entropy không trọng số để kết quả
có cùng cách diễn giải bất kể winner đến từ thí nghiệm nào


In [4]:
from src.evaluation.evaluator import evaluate
from src.evaluation.metrics import format_metrics_table

started = time.time()
test_metrics = evaluate(
    model, test_loader, device=device,
    loss_fn=torch.nn.CrossEntropyLoss(ignore_index=-100),
    use_amp=(device.type == "cuda"),
    desc=f"{WINNER} OFFICIAL TEST",
)
elapsed = time.time() - started

print(f"\nEvaluated {test_metrics['num_evaluated_tokens']:,} words in {elapsed:.1f}s\n")
print(format_metrics_table(test_metrics))

E2 OFFICIAL TEST:   0%|          | 0/715 [00:00<?, ?it/s]


Evaluated 2,968,815 words in 38.3s

label       precision    recall        f1     support
-----------------------------------------------------
O              0.9861    0.9861    0.9861   2,702,879
COMMA          0.7407    0.7054    0.7226     137,612
PERIOD         0.7948    0.8240    0.8091     111,606
QUESTION       0.7484    0.8528    0.7972      16,718
-----------------------------------------------------
accuracy       0.9663
macro F1       0.8288   (all 4 classes)
PUNCT-F1       0.7763   <-- model selection metric (COMMA/PERIOD/QUESTION)


## 3. Kết quả test

Bảng đầu trình bày precision, recall, F1 và support của từng lớp. Bảng
tiếp theo đặt metric test cạnh metric validation của winner để quan sát
mức chênh lệch giữa hai split


In [5]:
val_pc = selection["winner_validation_f1_per_class"]
rows = []
for label in LABELS:
    m = test_metrics["per_class"][label]
    rows.append({
        "label": label,
        "test_precision": round(m["precision"], 6),
        "test_recall": round(m["recall"], 6),
        "test_f1": round(m["f1"], 6),
        "test_support": int(m["support"]),
        "validation_f1": round(val_pc[label], 6),
        "test_minus_validation_f1": round(m["f1"] - val_pc[label], 6),
    })
per_class_df = pd.DataFrame(rows)
display(per_class_df)

headline = pd.DataFrame([{
    "metric": "Punctuation Macro-F1 (chính)",
    "test": round(test_metrics["punctuation_macro_f1"], 6),
    "validation": round(selection["winner_validation_punctuation_macro_f1"], 6),
}, {
    "metric": "Accuracy",
    "test": round(test_metrics["accuracy"], 6),
    "validation": round(selection["winner_validation_accuracy"], 6),
}, {
    "metric": "Macro-F1 (4 lớp)",
    "test": round(test_metrics["macro_f1"], 6),
    "validation": round(selection["winner_validation_macro_f1"], 6),
}])
headline["test - validation"] = (headline["test"] - headline["validation"]).round(6)
display(headline)

,label,test_precision,test_recall,test_f1,test_support,validation_f1,test_minus_validation_f1
0,O,0.986077,0.986122,0.986099,2702879,0.986302,-0.000203
1,COMMA,0.740725,0.705382,0.722621,137612,0.726973,-0.004352
2,PERIOD,0.794758,0.824033,0.809131,111606,0.810835,-0.001705
3,QUESTION,0.748438,0.852793,0.797215,16718,0.798345,-0.001129


,metric,test,validation,test - validation
0,Punctuation Macro-F1 (chính),0.776322,0.778718,-0.002396
1,Accuracy,0.966265,0.966725,-0.000460
2,Macro-F1 (4 lớp),0.828767,0.830614,-0.001847


## 4. Confusion matrix

Hàng của confusion matrix là nhãn đúng, cột là nhãn dự đoán theo thứ tự
`O`, `COMMA`, `PERIOD`, `QUESTION`.

| Cặp nhầm lẫn | Diễn giải |
|---|---|
| `PERIOD -> O` | Bỏ sót dấu chấm |
| `O -> PERIOD` | Kết thúc câu quá sớm |
| `COMMA -> O` | Bỏ sót dấu phẩy |
| `O -> COMMA` | Thêm dấu phẩy không cần thiết |
| `COMMA <-> PERIOD` | Nhầm mức độ của ranh giới câu |
| `QUESTION -> O` | Bỏ sót ranh giới câu hỏi |


In [6]:
cm = np.array(test_metrics["confusion_matrix"])
cm_df = pd.DataFrame(cm,
                     index=[f"gold_{l}" for l in LABELS],
                     columns=[f"pred_{l}" for l in LABELS])
display(cm_df)
print("Tỉ lệ theo hàng (%) — đường chéo chính là recall:")
display((cm_df.div(cm_df.sum(axis=1), axis=0) * 100).round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (data, title, fmt) in zip(axes, [
    (cm, "Confusion matrix — số lượng", "{:,}"),
    (cm / cm.sum(axis=1, keepdims=True) * 100, "Chuẩn hoá theo hàng (%)", "{:.1f}"),
]):
    im = ax.imshow(data, cmap="Blues", aspect="auto")
    ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(LABELS, rotation=30)
    ax.set_yticks(range(len(LABELS))); ax.set_yticklabels(LABELS)
    ax.set_xlabel("predicted"); ax.set_ylabel("gold"); ax.set_title(title)
    vmax = data.max()
    for i in range(len(LABELS)):
        for j in range(len(LABELS)):
            ax.text(j, i, fmt.format(data[i, j]), ha="center", va="center", fontsize=9,
                    color="white" if data[i, j] > vmax * 0.5 else "black")
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle(f"Official test confusion matrix — winner {WINNER}", fontsize=13)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "final_confusion_matrix.png", dpi=150)
plt.show()

,pred_O,pred_COMMA,pred_PERIOD,pred_QUESTION
gold_O,2665368,24647,10936,1928
gold_COMMA,27483,97069,12034,1026
gold_PERIOD,8789,9012,91967,1838
gold_QUESTION,1363,318,780,14257


Tỉ lệ theo hàng (%) — đường chéo chính là recall:


,pred_O,pred_COMMA,pred_PERIOD,pred_QUESTION
gold_O,98.61,0.91,0.40,0.07
gold_COMMA,19.97,70.54,8.74,0.75
gold_PERIOD,7.88,8.07,82.40,1.65
gold_QUESTION,8.15,1.90,4.67,85.28


<USER_HOME>/AppData\Local\Temp\ipykernel_7276\281079397.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(LABELS))
test_f1 = [test_metrics["per_class"][l]["f1"] for l in LABELS]
val_f1  = [val_pc[l] for l in LABELS]
ax.bar(x - 0.2, val_f1, 0.4, label="validation", color="#9ecae1")
ax.bar(x + 0.2, test_f1, 0.4, label="official test", color="#2a7fb8")
for xi, v in zip(x - 0.2, val_f1):
    ax.text(xi, v + 0.01, f"{v:.3f}", ha="center", fontsize=8)
for xi, v in zip(x + 0.2, test_f1):
    ax.text(xi, v + 0.01, f"{v:.3f}", ha="center", fontsize=8, fontweight="bold")
ax.axhline(test_metrics["punctuation_macro_f1"], color="#1a7a3c", ls="--",
           label=f"test PUNCT-F1 = {test_metrics['punctuation_macro_f1']:.4f}")
ax.set_xticks(x); ax.set_xticklabels(LABELS)
ax.set_ylabel("F1"); ax.set_ylim(0, 1.05)
ax.set_title(f"Per-class F1 — winner {WINNER}")
ax.grid(axis="y", alpha=0.3); ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "final_per_class_f1.png", dpi=150)
plt.show()

<USER_HOME>/AppData\Local\Temp\ipykernel_7276\3574055876.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Phân tích lỗi

Confusion matrix cho biết số lượng lỗi; phần này bổ sung ngữ cảnh bằng
các ví dụ thật cho từng cặp nhầm lẫn. Dự đoán chi tiết được thực hiện
trên 4.000 mẫu test đầu tiên để giới hạn thời gian xử lý

Các metric chính và confusion matrix phía trên vẫn được tính trên toàn
bộ test split. Mẫu con chỉ phục vụ phân tích định tính và thống kê lỗi
chi tiết


In [8]:
from src.evaluation.error_analysis import (
    analyze_errors, error_rows_for_csv, ERROR_CSV_HEADER, KEY_CONFUSION_PAIRS,
)
from src.evaluation.evaluator import predict_word_labels

N_ERROR_SAMPLE = 4000
error_subset = test_examples[:N_ERROR_SAMPLE]
subset_loader, _, _ = build_eval_dataloader(
    CKPT_DIR, error_subset, batch_size=32, encoder=encoder, model_type=model_type
)

predictions = predict_word_labels(model, subset_loader, device=device,
                                  use_amp=(device.type == "cuda"))
analysis = analyze_errors(error_subset, predictions, max_examples_per_pair=5)

print(f"Phân tích trên {analysis['num_examples_analyzed']:,} example "
      f"({analysis['total_positions']:,} từ)")
print(f"Tổng số lỗi: {analysis['total_errors']:,}  "
      f"(error rate {analysis['error_rate']:.4%})\n")

key_df = pd.DataFrame([{
    "gold": p["gold_label"], "pred": p["predicted_label"], "count": p["count"],
    "% of errors": round(100 * p["share_of_all_errors"], 2), "nghĩa": p["meaning"],
} for p in analysis["key_confusion_pairs"]])
display(key_df)

2026-08-10 06:01:18 | INFO    | src.data.dataset | PhoBERT dataset ready: {'num_examples': 4000, 'num_windows': 4005, 'num_words': 520335, 'num_subwords': 548194, 'windows_per_example': 1.0012, 'subwords_per_word': 1.0535, 'examples_split_into_multiple_windows': 5, 'max_window_length': 192}


Phân tích trên 4,000 example (520,335 từ)
Tổng số lỗi: 17,592  (error rate 3.3809%)



,gold,pred,count,% of errors,nghĩa
0,COMMA,O,4844,27.54,bỏ sót dấu phẩy — hai mệnh đề bị dính vào nhau
1,O,COMMA,4342,24.68,thêm dấu phẩy thừa — chèn ranh giới mệnh đề kh...
2,COMMA,PERIOD,2134,12.13,cắt câu quá mạnh — đáng lẽ chỉ là dấu phẩy
3,PERIOD,COMMA,1567,8.91,kết thúc câu quá yếu — hai câu bị nối thành câ...
4,PERIOD,O,1466,8.33,bỏ sót dấu chấm — hai câu bị nhập làm một (run...
5,O,PERIOD,1967,11.18,chấm câu quá sớm — một câu bị chẻ đôi
6,QUESTION,O,233,1.32,mất hẳn câu hỏi — lỗi nghiêm trọng nhất về mặt...
7,O,QUESTION,343,1.95,thêm dấu hỏi thừa — biến câu trần thuật thành ...


In [ ]:
top_pairs = [p for p in analysis["all_pairs"] if p["is_error"]][:4]
for pair in top_pairs:
    print(f"\n{'=' * 78}")
    print(f"{pair['gold_label']} → {pair['predicted_label']}   "
          f"({pair['count']:,} lỗi, {100 * pair['share_of_all_errors']:.1f}% tổng lỗi)")
    print(f"Nghĩa: {pair['meaning']}")
    print("=" * 78)
    for case in pair["representative_examples"][:3]:
        print(f"  [{case['example_id']} @ từ {case['word_index']}]")
        print(f"    {case['context'][:220]}")


COMMA → O   (4,844 lỗi, 27.5% tổng lỗi)
Nghĩa: bỏ sót dấu phẩy — hai mệnh đề bị dính vào nhau
  [test_000001 @ từ 109]
    … sách báo thấy mình giống sỏi **thận**[gold=COMMA',' pred=O''] trong khi bs bảo bình thường. …
  [test_000002 @ từ 82]
    … đến viêm thận mãn hoặc suy **thận**[gold=COMMA',' pred=O''] gây nguy hiểm và điều trị …
  [test_000003 @ từ 87]
    … gối lành lại. trong giai đoạn **này**[gold=COMMA',' pred=O''] bạn cần khám chuyên khoa để …

O → COMMA   (4,342 lỗi, 24.7% tổng lỗi)
Nghĩa: thêm dấu phẩy thừa — chèn ranh giới mệnh đề không có thật
  [test_000002 @ từ 30]
    … nhưng nếu bị viêm đường tiết **niệu**[gold=O'' pred=COMMA','] bạn nên đi làm xét nghiệm …
  [test_000005 @ từ 20]
    … ích mẫu thì xuống được 40 **ngày**[gold=O'' pred=COMMA','] nhưng dừng thuốc thì lại dài …
  [test_000006 @ từ 127]
    … những người có cơ địa dị **ứng**[gold=O'' pred=COMMA','] điều trị hết là khó khăn.

COMMA → PERIOD   (2,134 lỗi, 12.1% tổng lỗi)
Nghĩa: cắt câu quá mạnh — đáng lẽ 

## 6. Lưu kết quả đánh giá

Metric tổng hợp, kết quả theo lớp, confusion matrix và error analysis
được ghi vào `outputs/evaluation/`. `final_report.json` kết hợp thông
tin dữ liệu, thí nghiệm, quyết định chọn winner và official test thành
một artifact


In [10]:
from src.utils.environment import collect_environment
from src.utils.hashing import hash_jsonl_dataset

data_hashes = read_json(OUTPUTS_DIR / "data" / "data_hashes.json")

final_results = {
    "evaluated_at_utc": pd.Timestamp.utcnow().isoformat(),
    "split": "test",
    "split_file": "data/processed/test.jsonl",
    "winner": WINNER,
    "winner_model": selection["winner_model"],
    "winner_weight_mode": selection["winner_weight_mode"],
    "winner_best_epoch": selection["winner_best_epoch"],
    "checkpoint_path": CKPT_DIR.relative_to(PROJECT_ROOT).as_posix(),
    "selection_was_validation_only": True,
    "test_used_for_selection": False,
    "evaluation_runs_on_test": 1,
    "num_examples": len(test_examples),
    "num_evaluated_words": int(test_metrics["num_evaluated_tokens"]),
    "evaluation_seconds": round(elapsed, 1),
    "metrics": {
        "accuracy": test_metrics["accuracy"],
        "macro_f1": test_metrics["macro_f1"],
        "weighted_f1": test_metrics["weighted_f1"],
        "punctuation_macro_f1": test_metrics["punctuation_macro_f1"],
        "punctuation_micro_f1": test_metrics["punctuation_micro_f1"],
        "loss_unweighted": test_metrics["loss"],
        "per_class": {l: test_metrics["per_class"][l] for l in LABELS},
    },
    "validation_reference": {
        "punctuation_macro_f1": selection["winner_validation_punctuation_macro_f1"],
        "accuracy": selection["winner_validation_accuracy"],
        "macro_f1": selection["winner_validation_macro_f1"],
        "f1_per_class": val_pc,
    },
    "confusion_matrix": {"labels": LABELS, "rows_are_gold": True,
                         "matrix": test_metrics["confusion_matrix"]},
    "data_hashes": {k: v.get("content_sha256") for k, v in data_hashes.items()
                    if isinstance(v, dict)},
    "environment": collect_environment(),
}
write_json(EVALUATION_DIR / "final_test_results.json", final_results)

write_csv(EVALUATION_DIR / "final_test_per_class.csv",
          list(per_class_df.columns), per_class_df.values.tolist())

write_csv(EVALUATION_DIR / "final_test_confusion_matrix.csv",
          ["gold\\pred"] + [f"pred_{l}" for l in LABELS],
          [[f"gold_{LABELS[i]}"] + [int(v) for v in cm[i]] for i in range(len(LABELS))])

write_csv(EVALUATION_DIR / "final_error_analysis.csv",
          ERROR_CSV_HEADER, error_rows_for_csv(analysis, max_examples=3))

write_json(EVALUATION_DIR / "final_error_analysis.json", analysis)

for name in ("final_test_results.json", "final_test_per_class.csv",
             "final_test_confusion_matrix.csv", "final_error_analysis.csv",
             "final_error_analysis.json"):
    p = EVALUATION_DIR / name
    print(f"  [{'OK  ' if p.exists() else 'MISS'}] {p.relative_to(PROJECT_ROOT)}"
          f"  ({p.stat().st_size / 1024:,.1f} KB)")

  [OK  ] outputs\evaluation\final_test_results.json  (3.6 KB)
  [OK  ] outputs\evaluation\final_test_per_class.csv  (0.3 KB)
  [OK  ] outputs\evaluation\final_test_confusion_matrix.csv  (0.2 KB)
  [OK  ] outputs\evaluation\final_error_analysis.csv  (5.1 KB)
  [OK  ] outputs\evaluation\final_error_analysis.json  (37.2 KB)


In [ ]:
verification = read_json(EVALUATION_DIR / "training_verification.json")
manifest     = read_json(OUTPUTS_DIR / "data" / "data_source_manifest.json")
comparison   = read_json(EVALUATION_DIR / "validation_model_comparison.json")

final_report = {
    "project": "Vietnamese Punctuation Restoration",
    "generated_at_utc": pd.Timestamp.utcnow().isoformat(),
    "task": "word-level punctuation restoration (O / COMMA / PERIOD / QUESTION)",
    "dataset": {
        "name": manifest["dataset_name"],
        "url": manifest["repository_url"],
        "commit": manifest["verified_commit"],
        "license": manifest["license"],
        "official_split_preserved": True,
        "hashes": {k: v.get("content_sha256") for k, v in
                   read_json(OUTPUTS_DIR / "data" / "data_hashes.json").items()
                   if isinstance(v, dict)},
    },
    "experiments": [
        {k: r[k] for k in ("experiment_id", "model", "weight_mode", "best_epoch",
                           "validation_punctuation_macro_f1", "validation_accuracy",
                           "validation_macro_f1", "validation_loss_unweighted")}
        for r in comparison["rows"]
    ],
    "selection": {
        "split": selection["selection_split"],
        "metric": selection["selection_metric"],
        "tie_breaker": selection["tie_breaker"],
        "winner": WINNER,
        "winner_locked": True,
        "test_was_used_for_selection": False,
        "margin_over_runner_up": selection["margin_over_runner_up"],
        "runner_up": selection["runner_up"],
    },
    "winner": {
        "experiment_id": WINNER,
        "model": selection["winner_model"],
        "weight_mode": selection["winner_weight_mode"],
        "checkpoint_path": CKPT_DIR.relative_to(PROJECT_ROOT).as_posix(),
        "validation_punctuation_macro_f1": selection["winner_validation_punctuation_macro_f1"],
    },
    "official_test": {
        "punctuation_macro_f1": test_metrics["punctuation_macro_f1"],
        "accuracy": test_metrics["accuracy"],
        "macro_f1": test_metrics["macro_f1"],
        "per_class_f1": {l: test_metrics["per_class"][l]["f1"] for l in LABELS},
        "per_class_precision": {l: test_metrics["per_class"][l]["precision"] for l in LABELS},
        "per_class_recall": {l: test_metrics["per_class"][l]["recall"] for l in LABELS},
        "num_words": int(test_metrics["num_evaluated_tokens"]),
        "evaluation_runs": 1,
    },
    "error_analysis_summary": {
        "analyzed_examples": analysis["num_examples_analyzed"],
        "error_rate": analysis["error_rate"],
        "top_confusions": [
            {"gold": p["gold_label"], "predicted": p["predicted_label"], "count": p["count"]}
            for p in analysis["all_pairs"] if p["is_error"]
        ][:8],
    },
    "training_verification_passed": verification["passed"],
    "environment": collect_environment(),
    "limitations": [
        "Dataset là hội thoại tư vấn y tế tiếng Việt; hiệu năng trên văn bản pháp "
        "luật, tin tức hay khẩu ngữ đời thường có thể thấp hơn.",
        "Chỉ khôi phục 4 nhãn: O, COMMA, PERIOD, QUESTION. Không có dấu chấm than, "
        "hai chấm, chấm phẩy, ngoặc kép.",
        "Viết hoa khi hiển thị là rule-based (đầu câu, sau PERIOD/QUESTION), không "
        "phải mô hình capitalization.",
        "Có ~0.32-0.35% câu trùng text giữa các split — đặc tính của corpus gốc; "
        "split chính thức được giữ nguyên chứ không bị chỉnh sửa.",
        "Chưa tích hợp ASR/audio.",
    ],
}
write_json(EVALUATION_DIR / "final_report.json", final_report)
print("Written:", (EVALUATION_DIR / "final_report.json").relative_to(PROJECT_ROOT))

Written: outputs\evaluation\final_report.json


## 7. Hoàn tất đánh giá cuối

Cell cuối kiểm tra các artifact bắt buộc và in lại metric chính của
winner


In [12]:
expected = [
    EVALUATION_DIR / "final_test_results.json",
    EVALUATION_DIR / "final_test_per_class.csv",
    EVALUATION_DIR / "final_test_confusion_matrix.csv",
    EVALUATION_DIR / "final_error_analysis.csv",
    EVALUATION_DIR / "final_report.json",
    FIGURES_DIR / "final_confusion_matrix.png",
    FIGURES_DIR / "final_per_class_f1.png",
]
ok = all(p.exists() for p in expected)
for p in expected:
    print(f"  [{'OK  ' if p.exists() else 'MISS'}] {p.relative_to(PROJECT_ROOT)}")

print("\n" + "=" * 78)
print(f"OFFICIAL TEST RESULT — winner {WINNER} ({selection['winner_model']})")
print("=" * 78)
print(f"  Punctuation Macro-F1 : {test_metrics['punctuation_macro_f1']:.6f}")
print(f"  Accuracy             : {test_metrics['accuracy']:.6f}")
print(f"  Macro-F1 (4 lớp)     : {test_metrics['macro_f1']:.6f}")
for l in LABELS:
    m = test_metrics["per_class"][l]
    print(f"    {l:<9} P={m['precision']:.4f}  R={m['recall']:.4f}  F1={m['f1']:.4f}  "
          f"support={int(m['support']):,}")
print(f"\n  Winner chosen on validation only; test evaluated exactly once.")
print(f"  Artifacts complete: {ok}")
print("=" * 78)

  [OK  ] outputs\evaluation\final_test_results.json
  [OK  ] outputs\evaluation\final_test_per_class.csv
  [OK  ] outputs\evaluation\final_test_confusion_matrix.csv
  [OK  ] outputs\evaluation\final_error_analysis.csv
  [OK  ] outputs\evaluation\final_report.json
  [OK  ] outputs\figures\final_confusion_matrix.png
  [OK  ] outputs\figures\final_per_class_f1.png

OFFICIAL TEST RESULT — winner E2 (vinai/phobert-base-v2)
  Punctuation Macro-F1 : 0.776322
  Accuracy             : 0.966265
  Macro-F1 (4 lớp)     : 0.828767
    O         P=0.9861  R=0.9861  F1=0.9861  support=2,702,879
    COMMA     P=0.7407  R=0.7054  F1=0.7226  support=137,612
    PERIOD    P=0.7948  R=0.8240  F1=0.8091  support=111,606
    QUESTION  P=0.7484  R=0.8528  F1=0.7972  support=16,718

  Winner chosen on validation only; test evaluated exactly once.
  Artifacts complete: True
